In [25]:
from IPython.display import display
import pandas as pd
import numpy as np

import main_analysis as ma

# Load results

In [2]:
probe_csv = pd.read_csv(ma.BEST_LAYERS_CSV_PATH)
crossval_csv = pd.read_csv(ma.MODELSUBJECT_CORRELATIONS_CSV_PATH)
froi_fmri_model_map = {
    # fmri fROI : model fROI
    'FFA': 'FFA',
    'OFA': 'FFA',
    'PPA': 'PPA',
    'OPA': 'PPA',
    'RSC': 'PPA',
    'EBA': 'EBA',
    'FBA': 'EBA',
}
frois = ['FFA', 'PPA', 'EBA']
tthreshold = 7
model_localizer = 'grillspector'

models_df = []  # List to hold DataFrames for each fROI

In [26]:
probe_csv_selected = probe_csv[
    (probe_csv['fmri_froi'].isin(frois)) &
    (probe_csv['model_froi'].isin(frois)) &
    (probe_csv['fmri_tthreshold'] == tthreshold) &
    (probe_csv['model_tthreshold'] == tthreshold) &
    (probe_csv['model_localizer'] == model_localizer)
]
# Remove the row corresponding to model name resnet50imagenet21k
probe_csv_selected = probe_csv_selected[probe_csv_selected['model_name'] != 'resnet50imagenet21k']
# Group by the model_name and take the median of normed_pearsonr_median, ignore non-matching layers
# probe_csv_selected = probe_csv_selected.drop(columns=['model_layer'])
probe_csv_selected = probe_csv_selected.groupby('model_name')['normed_pearsonr_median'].median().reset_index()
# Sort the DataFrame by normed_pearsonr_median in descending order, for visualization purposes
probe_csv_selected = probe_csv_selected.sort_values(by='normed_pearsonr_median', ascending=False).reset_index(drop=True)
# Get the best, median, and worst model for this fROI based on normed_pearsonr_median
best_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmax()]
worst_model_info = probe_csv_selected.loc[probe_csv_selected['normed_pearsonr_median'].idxmin()]
# Find the median model using argsort
sorted_indices = np.argsort(probe_csv_selected['normed_pearsonr_median'].values)
median_idx = len(sorted_indices) // 2
median_model_info = probe_csv_selected.iloc[sorted_indices[median_idx]]

# Store the results in a DataFrame
models_df = [
    {
        'model_type': 'best',
        'model_name': best_model_info['model_name'],
        'probe_normed_pearsonr_median': best_model_info['normed_pearsonr_median'],
    }, 
    {
        'model_type': 'median',
        'model_name': median_model_info['model_name'],
        'probe_normed_pearsonr_median': median_model_info['normed_pearsonr_median'],
    },
    {
        'model_type': 'worst',
        'model_name': worst_model_info['model_name'],
        'probe_normed_pearsonr_median': worst_model_info['normed_pearsonr_median'],
    }
]
models_df = pd.DataFrame(models_df)
display(models_df)

,model_type,model_name,probe_normed_pearsonr_median
0,best,vitb16siglip2res224,0.843869
1,median,vitb16synclr,0.733681
2,worst,alexnetipclref01,0.465497


In [27]:
crossval_csv_selected = crossval_csv[
    (crossval_csv['fmri_froi'].isin(frois)) &
    (crossval_csv['model_froi'].isin(frois)) &
    (crossval_csv['fmri_tthreshold'] == tthreshold) &
    (crossval_csv['model_tthreshold'] == tthreshold) &
    (crossval_csv['model_localizer'] == model_localizer)
]
# Group by model_name, model_froi and fmri_froi and get the median of normed_pearsonr and normed_rdmr (across subjects)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name', 'model_froi', 'fmri_froi'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Group by the model_name and get the median of normed_pearsonr and normed_rdmr (across fROIs)
crossval_csv_selected = crossval_csv_selected.groupby(['model_name'])[['normed_pearsonr', 'normed_rdmr']].median().reset_index()
# Rename the columns to indicate that these are cross-validated metrics
crossval_csv_selected = crossval_csv_selected.rename(columns={
    'normed_pearsonr': 'crossval_normed_pearsonr',
    'normed_rdmr': 'crossval_normed_rdmr',
})
# Get the rank of all models in crossval_csv_selected separately for normed_pearsonr and normed_rdmr
crossval_csv_selected['crossval_rank_normed_pearsonr'] = crossval_csv_selected['crossval_normed_pearsonr'].rank(ascending=False, method='min')
crossval_csv_selected['crossval_rank_normed_rdmr'] = crossval_csv_selected['crossval_normed_rdmr'].rank(ascending=False, method='min')
# Merge models_df with crossval_csv_selected on model_name to get the cross-validated metrics for the best, median, and worst models
models_df_ = models_df.merge(crossval_csv_selected, on='model_name', how='left')
# Drop probe_normed_pearsonr_median, crossval_normed_pearsonr and crossval_normed_rdmr columns
# models_df_ = models_df_.drop(columns=['probe_normed_pearsonr_median', 'crossval_normed_pearsonr', 'crossval_normed_rdmr'])
display(models_df_)

,model_type,model_name,probe_normed_pearsonr_median,crossval_normed_pearsonr,crossval_normed_rdmr,crossval_rank_normed_pearsonr,crossval_rank_normed_rdmr
0,best,vitb16siglip2res224,0.843869,0.711801,0.481560,8.0,6.0
1,median,vitb16synclr,0.733681,0.696896,0.305190,12.0,27.0
2,worst,alexnetipclref01,0.465497,0.459088,0.268406,35.0,31.0
